# Unit 05 - Strategies and Challenges in Federated Learning

**Authors**:

- Patricia Guadalupe Alvarenga Mairena
- Laura González Lemos
- Iria Janeiro Pazos
- Ayesha Munir 
- Andrea Real Blanco

# Strategies and Challenges in Federated Learning

In the previous unit, we introduced the fundamentals of **Federated Learning** and implemented a complete end-to-end federated training pipeline using Flower. We focused on understanding the interaction between clients and server, the role of local training, and the basic Federated Averaging (FedAvg) algorithm.

In this unit, we move one step further. Rather than treating federated learning as a single algorithm, we explore it as a **family of strategies** designed to address practical challenges that arise in real-world deployments. These challenges include data heterogeneity across clients, limited communication budgets, partial client participation, and unstable convergence.

The main objective of this notebook is twofold:
1. To introduce and conceptually understand **alternative federated learning strategies beyond FedAvg**.
2. To analyze how these strategies relate to specific challenges and how they can be explored experimentally.

Throughout the notebook, we will deliberately reuse components from the previous unit. Some code fragments are intentionally left incomplete as short exercises, allowing you to refresh key concepts and actively engage with the design of federated learning systems.

## Recap Exercise: Rebuilding the Federated Learning Baseline (from Unit 04)

Before introducing new strategies, we briefly revisit the baseline federated learning pipeline implemented in the previous unit. The goal is to refresh the key components required to run a federated learning experiment in Flower:

- a client implementing the `NumPyClient` interface,
- a `ClientApp` that instantiates one client per partition,
- a `ServerApp` that defines the strategy and the number of rounds,
- and the simulation entrypoint.

Complete the following skeleton by filling in the missing pieces.

> **Important:** This code is intentionally incomplete. Do **not** run it as-is. Create a new code cell, copy the completed version there, and execute it.

In [1]:
!pip install -q -U "flwr[simulation]"

In [2]:
import numpy as np
import tensorflow as tf
from typing import List, Dict, Optional, Tuple, Union

import flwr as fl

NUM_CLIENTS = 10
NUM_ROUNDS = 5


# Load CIFAR-10 and partition it
def load_datasets(num_clients: int):
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    y_train = y_train.flatten()
    y_test = y_test.flatten()

    # Split training data into clients
    train_partitions = np.array_split(x_train, num_clients)
    label_partitions = np.array_split(y_train, num_clients)

    trainloaders = []
    valloaders = []

    for i in range(num_clients):
        x_part = train_partitions[i]
        y_part = label_partitions[i]

        split_idx = int(0.8 * len(x_part))
        x_local_train, x_local_val = x_part[:split_idx], x_part[split_idx:]
        y_local_train, y_local_val = y_part[:split_idx], y_part[split_idx:]

        trainloaders.append((x_local_train, y_local_train))
        valloaders.append((x_local_val, y_local_val))

    testloaders = (x_test, y_test)

    return trainloaders, valloaders, testloaders


trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)


# Define a simple neural network model
def generate_ann():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


### Part A — Client logic
class MyClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, val_data):
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        x_train, y_train = self.train_data
        self.model.fit(x_train, y_train, epochs=1, batch_size=32, verbose=0)

        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)

        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)

        return float(loss), len(x_val), {"accuracy": float(accuracy)}


### Part B — ClientApp
from flwr.common import Context
from flwr.clientapp import ClientApp

def client_fn(context: Context) -> fl.client.Client:
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)

    model = generate_ann()
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    return MyClient(model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)


### Part C — ServerApp
from flwr.server import ServerApp, ServerAppComponents

def server_fn(context: Context):
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    config = fl.server.ServerConfig(num_rounds=NUM_ROUNDS)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)


### Part D — Run experiment
from flwr.simulation import run_simulation

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


DEBUG:flwr:Asyncio event loop already running.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behav

### Reflection questions

1. Which parts of this pipeline run on the client side and which run on the server side?

2. What information is exchanged between server and clients at each round?

3. Why is it important to return the correct number of examples from fit and evaluate?

### Answers


1. On the **client side**, each client receives the global model, updates the model with its own local data, and evaluates it on its local validation set. In our notebook, this is mainly done inside the client class through `fit()` and `evaluate()`. On the **server side**, the server starts the federated process, selects clients, sends the current global parameters, collects their updates, and aggregates them to build the new global model. In our notebook, this is handled by the Flower strategy and the `ServerApp`.

2. At each round, the **server sends the current global model parameters** to the selected clients. Then each **client sends back updated model parameters** after local training, together with the **number of local examples used**. During evaluation, clients also send back their **loss/accuracy results** and the **number of validation examples**.

3. It is important because Flower uses the number of examples to do a **weighted aggregation**. This means clients with more data should contribute more to the global update than clients with less data. If we return the wrong number of examples, the server will combine the updates incorrectly, and the final global model may become unfair or less accurate. The same idea applies to evaluation, because the global metrics should reflect the real size of each client’s validation data.





It may be worth mentioning that Flower, by default, initializes the global model by making a call to one random client before distributing it to the remaining clients. However, sometimes more control is required, such as when performing fine-tuning. In such situations, we use server-side initialization, and the `initial_parameters` parameter will hold the initial version of the model for all clients. It is important to note that this parameter must be a serialization of the data, so the utility function `ndarrays_to_parameters` can be quite handy in this case.

## Federated Learning Strategies (and why custom strategies matter)

Federated learning is not defined by a single training algorithm. Instead, it comprises a **family of strategies** that specify how the server orchestrates training: which clients participate, what instructions they receive, how their updates are combined, and how the global model is updated across rounds.

In Flower, these choices are encapsulated in **server-side strategies** (e.g., `FedAvg`). Importantly, strategies are not limited to the ones provided by the library. In many real deployments, researchers and practitioners implement **custom strategies** to match the requirements of a specific application, such as:
- coping with highly non-IID data,
- handling unreliable or slow clients,
- improving convergence with server-side optimizers,
- enforcing fairness constraints or robust aggregation rules,
- or integrating privacy/security mechanisms.

In the next sections, we first introduce FedAvg as a baseline strategy, and then discuss representative alternatives. Throughout the notebook, keep in mind that each strategy can be interpreted as a different design choice about *how the server should use the information coming from clients*.


## Baseline Strategy: Federated Averaging (FedAvg)

The most widely used baseline in federated learning is **Federated Averaging (FedAvg)**. At a high level, FedAvg alternates between two steps:

1. The server sends the current global model parameters to a subset of clients.
2. Each selected client performs local training on its private data and returns updated parameters to the server.

The server then aggregates the client updates—typically using a **weighted average**, where each client’s contribution is proportional to the number of local training examples. This simple mechanism often works well when client data are reasonably similar, but its performance can degrade under strong heterogeneity (e.g., highly non-IID data or very different client behaviors).

In Flower, FedAvg is implemented as a server-side strategy. The main hyperparameters you can control include the fraction of clients participating per round, the minimum number of available clients, and (optionally) the initialization of the global model parameters. In the next sections, we will use FedAvg as a reference point to motivate and understand more advanced strategies.

This has should have been already experimented in the previous lesson and the recap of this one.

## Beyond FedAvg: Alternative and Custom Federated Strategies

While FedAvg provides a simple and effective baseline, many real-world federated learning scenarios violate its underlying assumptions. In particular, client datasets are often **non-IID**, clients may have very different computational capabilities, and only a subset of clients may be available at any given time. These issues motivate the development of **alternative federated learning strategies**.

Several extensions of FedAvg have been proposed to address these challenges. For example, **FedProx** introduces a regularization term in the local objective to prevent client models from drifting too far from the global model, which can improve stability under data heterogeneity. Other approaches, such as **FedOpt** methods (e.g., FedAdam, FedYogi), modify the server-side update rule by applying adaptive optimization techniques at the aggregation step instead of relying on simple averaging.

An important design principle in Flower is that strategies are **modular and extensible**. Researchers and practitioners are not restricted to predefined strategies: it is possible to implement **custom strategies** by subclassing the strategy interface and redefining how client updates are selected, aggregated, or interpreted. This flexibility is essential in practice, as many applications require domain-specific constraints, robustness mechanisms, or fairness-aware aggregation rules that go beyond standard algorithms.

In the following sections, we will connect these strategies to the main challenges in federated learning and explore how changing the server-side strategy impacts training dynamics and model performance.

## Centralized vs Federated Evaluation

So far, evaluation has been treated as part of the federated learning loop, but it is important to make explicit that **there are multiple ways to evaluate a federated model**, and that this choice is closely tied to the server-side strategy.

Broadly speaking, two evaluation paradigms can be distinguished: **centralized (server-side) evaluation** and **federated (client-side) evaluation**.

In **centralized evaluation**, the server evaluates the aggregated global model on a fixed dataset that is not used for training. This approach closely resembles traditional centralized machine learning. It has the advantage of producing stable and reproducible evaluation results, since the evaluation dataset is always the same. In addition, it avoids extra communication with clients during evaluation rounds. However, centralized evaluation assumes that the server has access to a representative evaluation dataset, which is not always realistic in privacy-sensitive or highly decentralized scenarios.

In **federated evaluation**, evaluation is performed by the clients using their local datasets. The server sends the current global model to the clients, each client evaluates it locally, and the resulting metrics are sent back and aggregated by the server. This approach often better reflects real-world federated deployments, as it relies exclusively on decentralized data. At the same time, it introduces additional challenges: evaluation results may fluctuate across rounds due to partial client participation, changing local datasets, or heterogeneous data distributions. Moreover, federated evaluation increases communication costs, since models must be transmitted to clients for evaluation.

The examples used so far in this notebook rely on **federated evaluation**, as they implement an `evaluate` method on the client side and aggregate the resulting metrics on the server. As we will see next, controlling how evaluation is performed and aggregated is another reason why customizing federated learning strategies is often necessary.


## Implementing Custom Strategies in Flower (Code Skeleton)

In Flower, a federated learning strategy is implemented on the **server side** and defines the overall orchestration of the learning process. While built-in strategies such as FedAvg cover many standard use cases, real-world federated learning systems often require **custom strategies** tailored to specific constraints or objectives.

Implementing a custom strategy typically involves extending an existing one (most commonly `FedAvg`) and overriding selected parts of its behavior. These extensions may affect how client updates are aggregated, how clients are sampled at each round, which configuration parameters are sent to clients (for example, the number of local epochs or steps per epoch), or how evaluation metrics are combined and reported.

The following code provides a minimal **skeleton of a custom FedAvg-like strategy**. It highlights the key extension points without introducing unnecessary complexity.

> **Exercise:** Complete the TODOs in the skeleton below and replace the baseline FedAvg strategy in the `ServerApp` with your custom implementation. Observe how changing server-side logic influences the training dynamics and evaluation results.


**Note**: This strategy template is based on Flower versions prior to 1.2. [Example](https://flower.ai/docs/framework/1.19/en/how-to-aggregate-evaluation-results.html)

In [3]:
from typing import List, Tuple, Optional, Dict, Any
import flwr as fl
from flwr.common import Metrics
from flwr.server.client_proxy import ClientProxy

class MyCustomStrategy(fl.server.strategy.FedAvg):
    """Example of a custom strategy extending FedAvg."""

    def __init__(
        self,
        local_epochs: int = 1,
        steps_per_epoch: Optional[int] = None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.local_epochs = local_epochs
        self.steps_per_epoch = steps_per_epoch

    def configure_fit(
        self,
        server_round: int,
        parameters: fl.common.Parameters,
        client_manager: fl.server.client_manager.ClientManager,
    ):
        """Send custom config to clients before local training."""
        fit_ins_list = super().configure_fit(server_round, parameters, client_manager)

        for client, fit_ins in fit_ins_list:
            fit_ins.config["local_epochs"] = self.local_epochs
            fit_ins.config["steps_per_epoch"] = (
                self.steps_per_epoch if self.steps_per_epoch is not None else 0
            )

        return fit_ins_list

    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, fl.common.EvaluateRes]],
        failures,
    ):
        """Aggregate evaluation accuracy using weighted average."""
        if not results:
            return None, {}

        total_examples = sum(res.num_examples for _, res in results)
        weighted_acc = sum(res.metrics["accuracy"] * res.num_examples for _, res in results)

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)
        return aggregated_loss, {"accuracy": weighted_acc / total_examples}

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Using the Custom Strategy in the ServerApp

Defining a custom strategy is only the first step. To actually use it in a federated learning experiment, we must instantiate it on the server side and return it from the `ServerApp`.

In the next code cell, replace the baseline `FedAvg` strategy with `MyCustomStrategy`. Then, run the simulation and verify two things:

1. The client receives the hyperparameters you injected through `configure_fit` (via the `config` dictionary).
2. The server reports an aggregated accuracy computed by your custom `aggregate_evaluate` implementation.

**Note:** If you do not modify the client’s `fit` method to read `local_epochs` and `steps_per_epoch` from `config`, changing `configure_fit` will have no effect.


In [4]:
# Example: instantiate and use the custom strategy in the ServerApp

def server_fn(context: Context):
    # Initialize global model parameters (recommended)
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    strategy = MyCustomStrategy(
        local_epochs=1,
        steps_per_epoch=3,
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    config = fl.server.ServerConfig(num_rounds=NUM_ROUNDS)
    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)

# Client-side logic adapted to receive custom configuration from the server
class MyClient(fl.client.NumPyClient):
    def __init__(self, model, train_data, val_data):
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        # Load global parameters
        self.model.set_weights(parameters)

        local_epochs = int(config.get("local_epochs", 1))
        steps_per_epoch = int(config.get("steps_per_epoch", 0))
        if steps_per_epoch == 0:
            steps_per_epoch = None

        x_train, y_train = self.train_data

        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        # Load global parameters
        self.model.set_weights(parameters)

        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)

        return float(loss), len(x_val), {"accuracy": float(accuracy)}

In [5]:
# Execute the simulation
history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

print("Custom strategy simulation finished.")
history

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=gcs_server) [2026-03-16 10:17:39,531 E 5273 5273] (gcs_server) gcs_server.cc:302: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzi

Custom strategy simulation finished.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Strategy Spotlight: FedProx (Stabilizing Training under Heterogeneity)

A major limitation of FedAvg is that it can become unstable when client data are strongly **non-IID**. In such cases, local updates may drift in very different directions, and simple averaging may slow down convergence or even lead to oscillations.

**FedProx** addresses this issue by modifying the *local* optimization objective. Instead of minimizing only the client’s local loss, FedProx adds a **proximal term** that penalizes large deviations from the current global model. Intuitively, this keeps local training “closer” to the global solution and can improve robustness under heterogeneity. This proximal term is typically controlled by a hyperparameter (often denoted as *μ*), which determines how strongly local updates are constrained.

From a practical perspective, FedProx can be seen as a strategy that changes what happens inside the client’s `fit` step: local training becomes more conservative when client distributions differ substantially. This makes FedProx a good example of an approach where the main “strategy change” is not only server-side aggregation, but also how local optimization is performed.

In the next section, we will simulate heterogeneity and discuss why strategies like FedProx (or custom variants) are often considered when FedAvg underperforms.

#### Advanced Note: Building a Strategy from Scratch

So far, we have customized federated learning behavior by **extending an existing strategy** (e.g., subclassing `FedAvg`). This is often the most practical approach, since it allows you to reuse a well-tested baseline and modify only the parts you need (sampling, configuration, aggregation, metrics).

In some research or engineering scenarios, however, it can be useful to implement a strategy **from scratch** by extending Flower’s strategy interface (`flwr.server.strategy.Strategy`). This gives full control over the server-side workflow, including how clients are selected, which instructions are sent each round, and how results are aggregated into a new global model.

Because implementing a full strategy requires more boilerplate and careful handling of edge cases (failures, partial participation, timeouts), we treat it as an advanced topic and focus on extending existing strategies in this notebook.

For an end-to-end example of a strategy built from scratch, see the official tutorial:
[Build a Strategy from Scratch](https://flower.ai/docs/framework/tutorial-series-build-a-strategy-from-scratch-pytorch.html#Build-a-Strategy-from-scratch)


# Challenges in Federated Learning

While federated learning addresses important limitations of centralized machine learning—such as data privacy and the need to move large datasets—it also introduces a set of **new challenges** that directly affect training dynamics and strategy design. In this section, we focus on one of the most fundamental of these challenges: **non-IID data**.

## Non-IID Data

In many machine learning settings, it is common to assume that data are **independent and identically distributed (IID)**. Under this assumption, each data sample is generated independently and follows the same underlying distribution. This assumption simplifies both theoretical analysis and practical algorithm design.

Federated learning, however, rarely satisfies this assumption. Because data are collected and stored locally on different devices, each client typically observes data drawn from a **different distribution**. This leads to **non-IID data**, where statistical properties vary significantly across clients.


![Diagram with IID and non-IID data](https://datasciences.org/wp-content/themes/dslabNew/images/datasciences/IIDness.png)
Credit: [Source of the image](https://datasciences.org/non-iid-learning/)


Non-IID data can arise for many reasons. Data may be correlated over time, influenced by user behavior, or biased toward specific classes or patterns. For example, one client may predominantly collect images of cats, while another may mostly contain images of dogs. As a result, local models trained on different clients may move in very different directions during optimization.

This heterogeneity poses a significant challenge for federated learning. Simple aggregation strategies such as FedAvg implicitly assume that local updates are roughly aligned. When this assumption is violated, training may become unstable, convergence may slow down, or the global model may oscillate between incompatible solutions.

From a strategy-design perspective, non-IID data is one of the main motivations for **alternative and custom federated learning strategies**. Approaches such as FedProx aim to constrain local updates, while custom aggregation rules may reweight or filter client contributions to reduce the impact of extreme heterogeneity. In Flower, addressing non-IID data typically involves modifying the **server-side strategy**, either by extending existing strategies or by implementing custom aggregation logic.

In the next section, we will see how such ideas can be translated into concrete strategy implementations.


## System and Device Heterogeneity

Beyond data heterogeneity, federated learning systems must also cope with **heterogeneity in hardware, software, and connectivity** across clients. Unlike centralized settings—where training typically runs on homogeneous clusters—federated learning involves devices with very different computational capabilities, memory limits, energy constraints, and network conditions.

In practice, this means that some clients may train much faster than others, some may only be intermittently available, and others may fail to complete local training altogether. These effects are often referred to as **system heterogeneity** and can significantly influence both efficiency and convergence.

From a learning perspective, system heterogeneity interacts with strategy design in several ways. Limiting the number of local epochs, adjusting `steps_per_epoch`, or sampling only a subset of clients per round are common techniques to reduce stragglers and stabilize training. From a systems perspective, strategies may also need to tolerate partial participation and client dropouts without compromising robustness.

In Flower, these issues are typically addressed at the **strategy level**, by controlling client sampling, configuring local workloads through the `config` dictionary, and deciding how to handle missing or failed client updates. In the following section, we will examine how communication constraints further shape these design choices.


In [6]:
import random
class HeterogeneousClient(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # Simulate system heterogeneity via different local computation budgets
        if self.cid == 0:
            local_epochs = 1
            steps_per_epoch = 1
        elif self.cid % 2 == 0:
            local_epochs = 3
            steps_per_epoch = 3
        else:
            local_epochs = 5
            steps_per_epoch = 5

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )

        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}


def client_fn(context: Context) -> fl.client.Client:
    partition_id = context.node_config.get("partition-id", context.node_id)
    cid = int(partition_id)

    model = generate_ann()
    train_data = trainloaders[cid]
    val_data = valloaders[cid]

    return HeterogeneousClient(cid, model, train_data, val_data).to_client()

client_app = ClientApp(client_fn=client_fn)

In [7]:
from typing import List, Tuple, Dict, Optional, Union
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy

class AggregateCustomMetricStrategy(fl.server.strategy.FedAvg):
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:
        """Aggregate evaluation accuracy using weighted average."""
        if not results:
            return None, {}

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)

        accuracies = [res.metrics["accuracy"] * res.num_examples for _, res in results]
        examples = [res.num_examples for _, res in results]

        aggregated_accuracy = sum(accuracies) / sum(examples)
        print(f"[Server] Round {server_round} aggregated accuracy: {aggregated_accuracy:.4f}")

        return aggregated_loss, {"accuracy": aggregated_accuracy}


from flwr.server import ServerApp, ServerAppComponents

def server_fn(context: Context):
    # Instantiate the model and create initial global parameters
    model = generate_ann()
    initial_parameters = fl.common.ndarrays_to_parameters(model.get_weights())
    del model

    strategy = AggregateCustomMetricStrategy(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=initial_parameters,
    )

    # Define ServerConfig
    config = fl.server.ServerConfig(num_rounds=5)

    return ServerAppComponents(strategy=strategy, config=config)

server_app = ServerApp(server_fn=server_fn)


from flwr.simulation import run_simulation

NUM_CLIENTS =10
trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

history = run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS,
)

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=8546) 2026-03-16 10:25:03.385562: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=8546) WARNING: All log messages before absl::InitializeLog() is

[Server] Round 1 aggregated accuracy: 0.1580


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 2 aggregated accuracy: 0.1587


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 4]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


...

INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 4 aggregated accuracy: 0.1959


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 400.28s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.2811026096343996
INFO :      		round 2: 2.2566976070404055
INFO :      		round 3: 2.2237971782684327
INFO :      		round 4: 2.1810181379318236
INFO :      		round 5: 2.1487056970596314
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 0.15799999982118607),
INFO :      	              (2, 0.15869999974966048),
INFO :      	              (3, 0.1721000000834465),
INFO :      	              (4, 0.19590000212192535),
INFO :      	              (5, 0.18789999783039094)]}
INFO :      


[Server] Round 5 aggregated accuracy: 0.1879


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Exercise: Exploring the Effect of System Heterogeneity

In the previous code, clients do not contribute equally: some clients perform more local computation (more epochs/steps), while others behave like resource-constrained devices. This is a simplified way of simulating **system and device heterogeneity**.

Complete the following tasks and compare the resulting training dynamics:

1. **Increase the heterogeneity gap**  
   Make one client extremely slow (e.g., `local_epochs=1`, `steps_per_epoch=1`) while keeping the rest as fast clients.

2. **Swap roles**  
   Make the previously “fast” clients slow, and the slow clients fast. Does the global performance change?

3. **Reduce participation**  
   Modify the server strategy so that only a fraction of clients participate each round, for example:
```python
    NUM_CLIENTS = 100
    strategy = fl.server.strategy.FedAvg(
        fraction_fit=0.025,  # Train on 25 clients (each round)
        fraction_evaluate=0.05,  # Evaluate on 50 clients (each round)
        min_fit_clients=20,
        min_evaluate_clients=40,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=fl.common.ndarrays_to_parameters(params),
        on_fit_config_fn=fit_config
    )
```
   How does partial participation interact with heterogeneous client speeds?

4. **Discussion**  
   Based on your observations, explain why system heterogeneity is not only a systems issue but also a learning issue.  
   Which strategy-level mechanisms could mitigate the negative effects of stragglers?

As you run these variations, keep track of:
- aggregated accuracy per round (printed by the server),
- stability across rounds (does accuracy fluctuate?),
- and the number of rounds needed to reach a comparable performance.


In [8]:
from typing import List, Tuple, Dict, Optional, Union
import flwr as fl
from flwr.common import EvaluateRes, FitRes, Scalar
from flwr.server.client_proxy import ClientProxy

class AggregateAccuracyStrategy(fl.server.strategy.FedAvg):
    def aggregate_evaluate(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, EvaluateRes]],
        failures: List[Union[Tuple[ClientProxy, FitRes], BaseException]],
    ) -> Tuple[Optional[float], Dict[str, Scalar]]:
        if not results:
            return None, {}

        aggregated_loss, _ = super().aggregate_evaluate(server_round, results, failures)

        accuracies = [res.metrics["accuracy"] * res.num_examples for _, res in results]
        examples = [res.num_examples for _, res in results]
        aggregated_accuracy = sum(accuracies) / sum(examples)

        print(f"[Server] Round {server_round} aggregated accuracy: {aggregated_accuracy:.4f}")

        return aggregated_loss, {"accuracy": aggregated_accuracy}

### Experiment 1: Increasing the heterogeneity gap

In this experiment, we make one client extremely slow while keeping the rest of the clients fast.

- **Slow client:** `local_epochs = 1`, `steps_per_epoch = 1`
- **Fast clients:** `local_epochs = 5`, `steps_per_epoch = 5`

The goal is to observe how a single highly constrained client affects the global training process. We will analyze the aggregated accuracy, the stability across rounds, and whether the model converges more slowly compared with a more balanced setup.

In [9]:
NUM_CLIENTS = 10
NUM_ROUNDS = 5

trainloaders, valloaders, testloaders = load_datasets(NUM_CLIENTS)

class HeterogeneousClientExp1(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # One extremely slow client, all others fast
        if self.cid == 0:
            local_epochs = 1
            steps_per_epoch = 1
        else:
            local_epochs = 5
            steps_per_epoch = 5

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )
        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}

def client_fn_exp1(context: Context) -> fl.client.Client:
    cid = int(context.node_config["partition-id"])
    model = generate_ann()
    return HeterogeneousClientExp1(cid, model, trainloaders[cid], valloaders[cid]).to_client()

client_app_exp1 = ClientApp(client_fn=client_fn_exp1)

def server_fn_exp1(context: Context):
    model = generate_ann()
    params = model.get_weights()

    strategy = AggregateAccuracyStrategy(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=fl.common.ndarrays_to_parameters(params),
    )

    config = fl.server.ServerConfig(num_rounds=NUM_ROUNDS)
    return ServerAppComponents(strategy=strategy, config=config)

server_app_exp1 = ServerApp(server_fn=server_fn_exp1)

history_exp1 = run_simulation(
    server_app=server_app_exp1,
    client_app=client_app_exp1,
    num_supernodes=NUM_CLIENTS,
)

print("Experiment 1 finished.")
history_exp1

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=11682) 2026-03-16 10:33:23.716789: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=11682) WARNING: All log messages before absl::InitializeLog() 

[Server] Round 1 aggregated accuracy: 0.1218


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 2 aggregated accuracy: 0.1815


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 4]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 3 aggregated accuracy: 0.2096


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 4 aggregated accuracy: 0.2417


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 404.51s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.316676068305969
INFO :      		round 2: 2.1947977781295775
INFO :      		round 3: 2.1052624225616454
INFO :      		round 4: 2.074960446357727
INFO :      		round 5: 2.0256400227546694
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 0.12180000096559525),
INFO :      	              (2, 0.1814999982714653),
INFO :      	              (3, 0.20960000157356262),
INFO :      	              (4, 0.24170000106096268),
INFO :      	              (5, 0.2737999975681305)]}
INFO :      


[Server] Round 5 aggregated accuracy: 0.2738
Experiment 1 finished.


### Experiment 2: Swapping client roles

In this experiment, we reverse the previous configuration.

- The client that was previously slow becomes fast
- The clients that were previously fast become slow

This helps us study whether the global model is more affected when most clients are slow rather than only one. We again compare the evolution of accuracy, the stability of training, and the number of rounds needed to reach a similar performance.

In [10]:
class HeterogeneousClientExp2(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # Swap roles: one fast client, rest slow
        if self.cid == 0:
            local_epochs = 5
            steps_per_epoch = 5
        else:
            local_epochs = 1
            steps_per_epoch = 1

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )
        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}

def client_fn_exp2(context: Context) -> fl.client.Client:
    cid = int(context.node_config["partition-id"])
    model = generate_ann()
    return HeterogeneousClientExp2(cid, model, trainloaders[cid], valloaders[cid]).to_client()

client_app_exp2 = ClientApp(client_fn=client_fn_exp2)

def server_fn_exp2(context: Context):
    model = generate_ann()
    params = model.get_weights()

    strategy = AggregateAccuracyStrategy(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
        initial_parameters=fl.common.ndarrays_to_parameters(params),
    )

    config = fl.server.ServerConfig(num_rounds=NUM_ROUNDS)
    return ServerAppComponents(strategy=strategy, config=config)

server_app_exp2 = ServerApp(server_fn=server_fn_exp2)

history_exp2 = run_simulation(
    server_app=server_app_exp2,
    client_app=client_app_exp2,
    num_supernodes=NUM_CLIENTS,
)

print("Experiment 2 finished.")
history_exp2

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)
(pid=16242) 2026-03-16 10:47:36.557416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=16242) WARNING: All log messages before absl::InitializeLog() 

[Server] Round 1 aggregated accuracy: 0.1202


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 2 aggregated accuracy: 0.1331


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 4]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


...

INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 10 clients (out of 10)


[Server] Round 4 aggregated accuracy: 0.1760


INFO :      aggregate_fit: received 10 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 404.51s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.3889554023742674
INFO :      		round 2: 2.311080551147461
INFO :      		round 3: 2.262282204627991
INFO :      		round 4: 2.239912176132202
INFO :      		round 5: 2.2451165914535522
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 0.12020000144839287),
INFO :      	              (2, 0.1331000007688999),
INFO :      	              (3, 0.1772000014781952),
INFO :      	              (4, 0.1760000020265579),
INFO :      	              (5, 0.1962000012397766)]}
INFO :      


[Server] Round 5 aggregated accuracy: 0.1962
Experiment 2 finished.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Experiment 3: Partial participation with heterogeneous clients

In this experiment, we keep heterogeneous clients, but only a fraction of them participate in each communication round.

This setting is closer to realistic federated learning scenarios, where not all devices are available all the time. The objective is to analyze how partial participation interacts with system heterogeneity and whether it makes training less stable or slows down convergence.

In [11]:
NUM_CLIENTS_PP = 10
NUM_ROUNDS_PP = 5

trainloaders_pp, valloaders_pp, testloaders_pp = load_datasets(NUM_CLIENTS_PP)

class HeterogeneousClientExp3(fl.client.NumPyClient):
    def __init__(self, cid, model, train_data, val_data):
        self.cid = cid
        self.model = model
        self.train_data = train_data
        self.val_data = val_data

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        self.model.set_weights(parameters)

        # A few slow clients, most clients fast
        if self.cid < 5:
            local_epochs = 1
            steps_per_epoch = 1
        else:
            local_epochs = 5
            steps_per_epoch = 5

        x_train, y_train = self.train_data
        self.model.fit(
            x_train,
            y_train,
            epochs=local_epochs,
            batch_size=32,
            steps_per_epoch=steps_per_epoch,
            verbose=0,
        )
        return self.model.get_weights(), len(x_train), {}

    def evaluate(self, parameters, config):
        self.model.set_weights(parameters)
        x_val, y_val = self.val_data
        loss, accuracy = self.model.evaluate(x_val, y_val, verbose=0)
        return float(loss), len(x_val), {"accuracy": float(accuracy)}

def client_fn_exp3(context: Context) -> fl.client.Client:
    cid = int(context.node_config["partition-id"])
    model = generate_ann()
    return HeterogeneousClientExp3(cid, model, trainloaders_pp[cid], valloaders_pp[cid]).to_client()

client_app_exp3 = ClientApp(client_fn=client_fn_exp3)

def server_fn_exp3(context: Context):
    model = generate_ann()
    params = model.get_weights()

    strategy = AggregateAccuracyStrategy(
        fraction_fit=0.25,
        fraction_evaluate=0.5,
        min_fit_clients=5,
        min_evaluate_clients=10,
        min_available_clients=NUM_CLIENTS_PP,
        initial_parameters=fl.common.ndarrays_to_parameters(params),
    )

    config = fl.server.ServerConfig(num_rounds=NUM_ROUNDS_PP)
    return ServerAppComponents(strategy=strategy, config=config)

server_app_exp3 = ServerApp(server_fn=server_fn_exp3)

history_exp3 = run_simulation(
    server_app=server_app_exp3,
    client_app=client_app_exp3,
    num_supernodes=NUM_CLIENTS_PP,
)

print("Experiment 3 finished.")
history_exp3

INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 5 clients (out of 10)
(pid=19055) 2026-03-16 10:54:36.755480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=19055) WARNING: All log messages before absl::InitializeLog() i

[Server] Round 1 aggregated accuracy: 0.1379


(ClientAppActor pid=19055) WARNING:tensorflow:5 out of the last 36 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x7e036e01aca0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 3]
INFO :      configure_fit: s

[Server] Round 2 aggregated accuracy: 0.1166


INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 4]
INFO :      configure_fit: strategy sampled 5 clients (out of 10)


...

INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [ROUND 5]
INFO :      configure_fit: strategy sampled 5 clients (out of 10)


[Server] Round 4 aggregated accuracy: 0.1701


INFO :      aggregate_fit: received 5 results and 0 failures
INFO :      configure_evaluate: strategy sampled 10 clients (out of 10)
INFO :      aggregate_evaluate: received 10 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 5 round(s) in 297.43s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.271374535560608
INFO :      		round 2: 2.3363686084747313
INFO :      		round 3: 2.238978147506714
INFO :      		round 4: 2.2070505380630494
INFO :      		round 5: 2.1838116884231566
INFO :      	History (metrics, distributed, evaluate):
INFO :      	{'accuracy': [(1, 0.13790000081062317),
INFO :      	              (2, 0.11660000011324882),
INFO :      	              (3, 0.16980000138282775),
INFO :      	              (4, 0.17010000050067903),
INFO :      	              (5, 0.18819999992847442)]}
INFO :      


[Server] Round 5 aggregated accuracy: 0.1882
Experiment 3 finished.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Exercise: Exploring the Effect of System Heterogeneity

In this exercise, we studied how different client speeds affect federated learning. We simulated heterogeneity by changing the amount of local computation performed by each client before sending updates to the server.

For practical reasons, we ran the experiments with **10 clients** and **5 communication rounds**. We reduced the scale of the simulation because larger settings were causing memory issues in the Colab environment. Even with this smaller setup, the experiments were still useful to observe the main effects of system heterogeneity.

#### Experiment 1: Increase the heterogeneity gap

In the first experiment, we made one client extremely slow by setting `local_epochs = 1` and `steps_per_epoch = 1`, while the rest of the clients were configured as fast clients with `local_epochs = 5` and `steps_per_epoch = 5`.

The goal was to check whether one strongly constrained client could negatively affect the global model. From the results, we observed that the model still learned in a relatively stable way. The aggregated accuracy improved from **0.1218** in round 1 to **0.2738** in round 5. This means that one slow client did not prevent learning, because most of the clients were still able to perform stronger local training.

#### Experiment 2: Swap roles

In the second experiment, we reversed the situation. The previously slow client became fast, and the other clients were made slow.

This configuration had a stronger negative effect on training. Since most clients were now doing only a small amount of local work, the global model improved more slowly. The aggregated accuracy increased from **0.1202** in round 1 to only **0.1962** in round 5. Compared with Experiment 1, the final performance was lower and the improvement across rounds was weaker. This shows that the number of slow clients matters more than the presence of a single slow client.

#### Experiment 3: Reduce participation

In the third experiment, we introduced partial participation so that only a fraction of clients trained in each round, while evaluation still used more clients. At the same time, some clients remained slow and others fast.

In our implementation, we used:
- `fraction_fit = 0.25`
- `fraction_evaluate = 0.5`
- `min_fit_clients = 5`
- `min_evaluate_clients = 10`

This setting produced more fluctuation across rounds. The aggregated accuracy started at **0.1379**, dropped to **0.1166** in round 2, and then increased to **0.1882** in round 5. This behavior suggests that partial participation and heterogeneity together make the training process less stable. Since only some clients contribute in each round, the quality of the global update depends more on which clients are selected.

#### Summary table

| Experiment | Setting | Final accuracy | Stability | Observation |
|---|---|---:|---|---|
| Exp 1 | One very slow client, rest fast | 0.2738 | Relatively stable | The model learned well because most clients were still strong contributors |
| Exp 2 | One fast client, rest slow | 0.1962 | Moderate | Training improved more slowly because most clients performed weak local updates |
| Exp 3 | Partial participation + heterogeneous clients | 0.1882 | Less stable | Accuracy fluctuated more because only part of the clients participated each round |

#### Discussion

These experiments show that system heterogeneity is not only a systems issue, but also a learning issue. At first sight, it may seem like a hardware or runtime problem, because some devices are slower than others. However, it also directly changes the quality of the updates sent to the server.

Clients that train more perform stronger local optimization, while slower clients send weaker updates. If this imbalance becomes large, the global model may converge more slowly, fluctuate more, or become biased toward the clients contributing stronger updates more often.

This effect becomes even more visible when only part of the clients participate in each round. In that case, the learning process depends even more on which clients are selected. If the sampled clients are mostly weak or slow, improvement may be limited during that round.

#### Strategy-level mechanisms to reduce the effect of stragglers

Some strategy-level mechanisms can help reduce the negative effect of stragglers:

- selecting clients more carefully instead of sampling completely at random,
- limiting the difference in local computation across clients,
- weighting updates properly according to the number of local examples,
- using partial participation with enough clients per round,
- dropping very slow clients when they delay the process too much,
- or adapting the strategy so that stale or weak updates have less impact.

Overall, we observed that heterogeneity changes not only the execution speed of the system, but also the behavior of the learning process itself.

## From Challenges to Strategies: A Practical Mapping

At this point, we have seen that *challenges* in federated learning are not just abstract limitations: they directly motivate *strategy choices*. A useful way to reason about federated learning is to ask:

**Which challenge is dominant in this scenario, and which strategy design choice addresses it?**

For example, strong non-IID data often motivates strategies that stabilize local training (e.g., FedProx-like ideas), system heterogeneity motivates partial participation or workload control (e.g., limiting steps per epoch), and communication constraints motivate reducing the number of rounds or compressing updates.
